In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import warnings

warnings.filterwarnings("ignore")

In [0]:
def compute_drift_metrics():
    w=Window().orderBy(F.col('obs_date')).rowsBetween(-11, 0)
    pred_df=spark.table('us_macroeconomics_tracker.gold.recession_predictions')
    final_df=pred_df.withColumn('rolling_avg_12m',F.avg(F.col('recession_probability')).over(w)).withColumn('std_dev_12m',F.stddev(F.col('recession_probability')).over(w))
    return final_df

In [0]:
def flag_drift():
    final_df=compute_drift_metrics()
    final_df=final_df.withColumn('drift_flag',F.when(F.abs(F.col('recession_probability') - F.col('rolling_avg_12m')) > 2 * F.col('std_dev_12m'),True).otherwise(False))
    return final_df

In [0]:
def save_monitoring_log():
    final_df=flag_drift()
    final_df=final_df.select('obs_date','recession_probability','rolling_avg_12m','std_dev_12m','drift_flag',F.current_timestamp().alias('checked_at'))
    final_df.write.format("delta").mode("overwrite").saveAsTable('us_macroeconomics_tracker.gold.monitoring_log')
save_monitoring_log()

In [0]:
print('Saving Monitoring Log...')